# SurvivalNet Demo

This notebook demonstrates an end-to-end workflow using the TCGA expression matrix and the paired clinical survival table.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from survivalnet import CoxModel, fit_km, plot_grouped_km_with_pvalue, split_risk_group
from survivalnet.models import LassoCoxModel

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 20)

EXPR_FILENAME = "TCGA.LUAD.sampleMap.txt"
CLINICAL_FILENAME = "LUAD_survival.txt"
EXAMPLE_DIR = Path("examples")


def find_existing_file(*relative_paths: Path) -> Path:
    """Return the first existing path from a list of candidates."""
    for candidate in relative_paths:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find any of: {', '.join(str(path) for path in relative_paths)}"
    )


expr_path = find_existing_file(
    EXAMPLE_DIR / EXPR_FILENAME,
    Path(EXPR_FILENAME),
    Path("../examples") / EXPR_FILENAME,
)
clinical_path = find_existing_file(
    EXAMPLE_DIR / CLINICAL_FILENAME,
    Path(CLINICAL_FILENAME),
    Path("../examples") / CLINICAL_FILENAME,
)

print(f"Using expression file: {expr_path.resolve()}")
print(f"Using clinical file: {clinical_path.resolve()}")

In [ ]:
def load_expression_matrix(path: Path) -> pd.DataFrame:
    """Load TCGA expression matrix and convert it to sample-by-gene layout."""
    expr = pd.read_csv(path, sep="\t", index_col=0).T.reset_index(names="PATIENT_ID")
    return expr


def load_clinical_table(path: Path) -> pd.DataFrame:
    """Load LUAD survival data and normalize the key columns."""
    clinical = pd.read_csv(path, sep="\t").rename(
        columns={"sample": "PATIENT_ID", "OS.time": "duration", "OS": "event"}
    )
    clinical = clinical[["PATIENT_ID", "duration", "event"]].dropna().copy()
    clinical["duration"] = (clinical["duration"] / 30.44).round(2)
    clinical["event"] = pd.to_numeric(clinical["event"], errors="coerce")
    clinical = clinical.dropna(subset=["event"])
    clinical["event"] = clinical["event"].astype(int)
    return clinical


expr_df = load_expression_matrix(expr_path)
clinical_df = load_clinical_table(clinical_path)
data = clinical_df.merge(expr_df, on="PATIENT_ID", how="inner")

print("Data loaded successfully")
print(f"Expression matrix: {expr_df.shape}")
print(f"Clinical table: {clinical_df.shape}")
print(f"Merged analysis set: {data.shape}")

In [ ]:
# Quick sanity check
display(data[["PATIENT_ID", "duration", "event"]].head())
print(f"Merged rows: {len(data)}")
print(f"Merged columns: {len(data.columns)}")

In [ ]:
# Kaplan-Meier curve for the whole cohort
kmf = fit_km(data, "duration", "event")
ax = kmf.plot_survival_function(figsize=(6, 4))
ax.set_title("Overall Survival - Kaplan-Meier")
ax.set_xlabel("Time (months)")
ax.set_ylabel("Survival probability")
plt.show()

In [ ]:
# Build a simple univariate Cox model with one representative gene
demo_gene = next(gene for gene in expr_df.columns if gene != "PATIENT_ID")
cox_df = data[["duration", "event", demo_gene]].copy()
cox_df[demo_gene] = pd.to_numeric(cox_df[demo_gene], errors="coerce")
cox_df = cox_df.dropna(subset=[demo_gene])

print(f"Rows used for Cox model: {len(cox_df)}")

cox_model = CoxModel().fit(cox_df, "duration", "event")

display(cox_model.summary)
display(cox_model.hazard_ratios)
risk_scores = cox_model.predict_risk_score(cox_df)
cox_df["risk_group"] = split_risk_group(risk_scores)
ax = plot_grouped_km_with_pvalue(
    cox_df,
    duration_col="duration",
    event_col="event",
    group_col="risk_group",
    group_a="low",
    group_b="high",
)
ax.set_title(f"High vs Low Risk Groups ({demo_gene})")
plt.show()

In [ ]:
gene_cols = [col for col in expr_df.columns if col != "PATIENT_ID"]
expr_lasso = expr_df.copy()
expr_lasso[gene_cols] = np.log2(expr_lasso[gene_cols] + 1)

expression_rate = (expr_lasso[gene_cols] > 0).mean(axis=0)
kept_genes = expression_rate[expression_rate >= 0.1].index.tolist()
top_genes = expr_lasso[kept_genes].var(numeric_only=True).sort_values(ascending=False).head(200).index.tolist()

lasso_df = (
    data.set_index("PATIENT_ID")[["duration", "event"]]
    .join(expr_lasso.set_index("PATIENT_ID")[top_genes], how="inner")
    .reset_index()
)

for gene in top_genes:
    lasso_df[gene] = pd.to_numeric(lasso_df[gene], errors="coerce")
    lasso_df[gene] = lasso_df[gene].fillna(lasso_df[gene].median())

print(f"Rows used for LASSO-Cox model: {len(lasso_df)}")
print(f"Features used: {len(top_genes)}")

sex_chromosome_genes = ["EIF1AY", "HY", "UTY", "USP9Y", "TTTY15", "CYorf15B", "DDX3Y", "RPS4Y1"]
penalizer_grid = [0.1, 0.05, 0.02, 0.01]
lasso_model = None
selected_features = []

for penalizer in penalizer_grid:
    try:
        candidate_model = LassoCoxModel(penalizer=penalizer).fit(lasso_df, "duration", "event")
        candidate_features = [gene for gene in candidate_model.selected_features if gene not in sex_chromosome_genes]
        if 8 <= len(candidate_features) <= 25:
            lasso_model = candidate_model
            selected_features = candidate_features
            print(f"Best penalizer: {penalizer} | selected features: {len(selected_features)}")
            break
    except Exception as exc:
        print(f"penalizer={penalizer} failed: {exc}")

if lasso_model is None:
    fallback_penalizer = 0.005
    print(f"Falling back to penalizer={fallback_penalizer}")
    lasso_model = LassoCoxModel(penalizer=fallback_penalizer).fit(lasso_df, "duration", "event")
    selected_features = [gene for gene in lasso_model.selected_features if gene not in sex_chromosome_genes]

print("Selected features:", selected_features)
display(lasso_model.fitter.summary.loc[selected_features])

lasso_df["risk_group"] = split_risk_group(lasso_model.predict_risk_score(lasso_df))
ax = plot_grouped_km_with_pvalue(
    lasso_df,
    duration_col="duration",
    event_col="event",
    group_col="risk_group",
    group_a="low",
    group_b="high",
)
ax.set_title("LASSO-Cox High vs Low Risk Groups (TCGA-LUAD)")
plt.show()